In [1]:
import core_config

print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")

[CORE] Secrets & Config loaded successfully from /nlsasfs/home/lmedge/rahkus/.env
[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.


In [ ]:
!python hf_upload_daemon.py 

[CORE] Secrets & Config loaded successfully from /nlsasfs/home/lmedge/rahkus/.env

[WARNING] The Hugging Face Dataset repository 'strikerhell/Vault' does NOT exist.
Would you like to create this repository now? (y/n): 

In [1]:
!quota_check

/bin/bash: line 1: quota_check: command not found


In [2]:
!squeue -u rahkus

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
            477065     dgxnp     bash   rahkus  R 1-06:41:03      1 scn14-10g
            474325     dgxnp     bash   rahkus  R 4-16:14:53      1 scn11-10g


In [1]:
!nvidia-smi

Wed Jul 22 13:24:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   30C    P0             56W /  400W |    5778MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# Compute-node test configuration.
import os
import core_config

core_config.map_storage()
core_config.set_hpc_proxies()

TESTING_MODE = True
ENABLE_CLOUD_UPLOAD = True
UPLOAD_TRIGGER_TYPE = "BATCH"
BATCH_LIMIT = 100

print("--- MASTER CONFIGURATION ---")
print(f"Interactive UI Mode: {TESTING_MODE}")
print(f"Cloud Upload: {ENABLE_CLOUD_UPLOAD} (Trigger: {UPLOAD_TRIGGER_TYPE})")
print(f"Models -> {core_config.SYSTEM_DIRS['models']}")
print(f"Cache -> {core_config.SYSTEM_DIRS['cache']}")
print(f"Outputs -> {core_config.SYSTEM_DIRS['outputs']}")
print("Compute-node configuration loaded. No model/package installation performed in this cell.")


Directories Mapped:
Models -> /nlsasfs/home/lmedge/rahkus/aireos_models
Datasets -> /nlsasfs/home/lmedge/rahkus/aireos_dataset
Cache -> /nlsasfs/home/lmedge/rahkus/aireos_cache

Ensuring buggy Rust downloader is disabled...
Proxies established! Compute node now has internet access.

[SECURE AUTHENTICATION REQUIRED]
Please paste your Hugging Face 'Write' token below. It will be masked.


KeyboardInterrupt: Interrupted by user

In [ ]:
# CELL 1: MASTER CONFIG & DEPENDENCY RESOLUTION
import os
import sys
from storage_manager import storage_preflight_path

# Package installation can consume several GB of quota before model files are present.
ok, reason = storage_preflight_path(
    core_config.SYSTEM_DIRS["cache"],
    required_gb=float(os.environ.get("AIEROS_ENV_INSTALL_BUDGET_GB", "6.0")),
    require_fresh_quota=True,
    minimum_free_gb=1.0,
)
if not ok:
    raise RuntimeError(f"Environment installation denied by storage policy: {reason}")


# --- 1. Master Configuration ---
TESTING_MODE = True           # True: Renders visual/audio outputs directly in Jupyter Notebook
ENABLE_CLOUD_UPLOAD = True    # True: Triggers HuggingFace batch uploads
UPLOAD_TRIGGER_TYPE = "BATCH" # Options: "BATCH" (every 100 assets), "HOURLY", "DAILY"
BATCH_LIMIT = 100             

print(f"--- MASTER CONFIGURATION ---")
print(f"Interactive UI Mode: {TESTING_MODE}")
print(f"Cloud Upload: {ENABLE_CLOUD_UPLOAD} (Trigger: {UPLOAD_TRIGGER_TYPE})")

# --- 2. The AudioCraft Official Fix (FFMPEG < 5) ---
print("\nInstalling official Conda dependencies for AudioCraft (ffmpeg<5 and av=11.0.0)...")
!conda install -c conda-forge "ffmpeg<5" "av=11.0.0" pkg-config -y

# --- 3. Dependency Installation (Strict PyTorch 2.1.0 + CUDA 11.8) ---
print("\nInstalling official PyTorch 2.1.0 (CUDA 11.8)...")
!{sys.executable} -m pip install "torch==2.1.0+cu118" "torchvision==0.16.0+cu118" "torchaudio==2.1.0+cu118" --index-url https://download.pytorch.org/whl/cu118

print("Installing core Python dependencies...")
!{sys.executable} -m pip install diffusers transformers accelerate imageio[ffmpeg] easydict huggingface_hub sentencepiece

print("Installing AudioCraft...")
!{sys.executable} -m pip install -U audiocraft

In [ ]:
# TEST A: LTX-VIDEO
import os
import traceback

import torch
from IPython.display import Video, display
from diffusers import LTXPipeline
from diffusers.utils import export_to_video
from storage_manager import managed_storage_operation
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes

MODEL_ID = "Lightricks/LTX-Video"
OUTPUT_DIR = os.path.join(core_config.SYSTEM_DIRS["outputs"], "tests")
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    print(f"\nCUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"Using Device: {torch.cuda.get_device_name(0)} (CUDA v{torch.version.cuda})")

    print("Verifying LTX-Video weights in local cache...")
    try:
        pipe = LTXPipeline.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.bfloat16,
            local_files_only=True,
        ).to("cuda")
        print("LTX cache hit; no model download required.")
    except Exception:
        required_gb = (
            estimate_hf_repo_bytes(MODEL_ID) / BYTES_PER_GB
            + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))
        )
        print(f"LTX cache incomplete. Storage admission budget: {required_gb:.2f} GB")
        with managed_storage_operation(
            "models",
            required_gb=required_gb,
            require_fresh_quota=True,
            minimum_free_gb=float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0")),
        ):
            pipe = LTXPipeline.from_pretrained(
                MODEL_ID,
                torch_dtype=torch.bfloat16,
            ).to("cuda")

    prompt = "A high-quality macro shot of a cyberpunk circuit board glowing with neon blue lights, slow pan."
    print(f"Generating video for prompt: '{prompt}'...")
    video = pipe(prompt=prompt, num_frames=81, width=704, height=480).frames[0]

    output_path = os.path.join(OUTPUT_DIR, "ltx_test_output.mp4")
    with managed_storage_operation(
        "outputs",
        required_gb=0.25,
        require_fresh_quota=False,
        minimum_free_gb=0.05,
    ):
        export_to_video(video, output_path, fps=24)

    print(f"Success! Video saved locally to {output_path}")
    if TESTING_MODE and os.path.exists(output_path):
        display(Video(output_path, embed=True, width=704))

except Exception:
    print("\n[ERROR IN LTX PIPELINE]")
    traceback.print_exc()


In [ ]:
# Wan2.1 source checkout is handled by Test B below.
# This obsolete raw `git clone` cell is intentionally disabled so repositories
# are kept under the OEC-managed cache and quota checks are applied first.
print("Use TEST B (next cell) for the quota-aware Wan2.1 source/model setup.")


In [ ]:
# TEST B: WAN 2.1 (Compute Node)
import os
import subprocess
import traceback
from IPython.display import Video, display
from storage_manager import managed_storage_operation
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes

WAN_REPO_DIR = os.path.join(
    core_config.SYSTEM_DIRS["cache"],
    "third_party",
    "Wan2.1",
)
WAN_MODEL_DIR = os.path.join(WAN_REPO_DIR, "Wan2.1-T2V-14B")
OUTPUT_DIR = os.path.join(core_config.SYSTEM_DIRS["outputs"], "tests")
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, "wan_test_output.mp4")

try:
    os.makedirs(os.path.dirname(WAN_REPO_DIR), exist_ok=True)

    if not os.path.exists(os.path.join(WAN_REPO_DIR, ".git")):
        repo_budget_gb = float(os.environ.get("AIEROS_WAN_REPO_BUDGET_GB", "3.0"))
        ok, reason = storage_preflight_path(
            core_config.SYSTEM_DIRS["cache"],
            required_gb=repo_budget_gb,
            require_fresh_quota=True,
            minimum_free_gb=0.5,
        )
        if not ok:
            raise RuntimeError(f"Wan2.1 source checkout denied by storage policy: {reason}")
        subprocess.run(
            ["git", "clone", "https://github.com/Wan-Video/Wan2.1.git", WAN_REPO_DIR],
            check=True,
        )

    if not os.path.exists(WAN_MODEL_DIR):
        model_budget_gb = (
            estimate_hf_repo_bytes("Wan-AI/Wan2.1-T2V-14B") / BYTES_PER_GB
            + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))
        )
        print(f"Downloading Wan2.1 model with storage budget: {model_budget_gb:.2f} GB")
        ok, reason = storage_preflight_path(
            core_config.SYSTEM_DIRS["cache"],
            required_gb=model_budget_gb,
            require_fresh_quota=True,
            minimum_free_gb=2.0,
        )
        if not ok:
            raise RuntimeError(f"Wan2.1 model download denied by storage policy: {reason}")

        subprocess.run(
            [
                "huggingface-cli",
                "download",
                "Wan-AI/Wan2.1-T2V-14B",
                "--local-dir",
                WAN_MODEL_DIR,
            ],
            check=True,
        )

    cmd = [
        sys.executable, "generate.py",
        "--task", "t2v-14B",
        "--size", "1280*720",
        "--ckpt_dir", "./Wan2.1-T2V-14B",
        "--prompt", "Cinematic drone shot over a glowing futuristic city at night.",
        "--save_file", output_path,
    ]

    print("Running Wan2.1 CLI inference...")
    with managed_storage_operation(
        "outputs",
        required_gb=1.0,
        require_fresh_quota=False,
        minimum_free_gb=0.05,
    ):
        result = subprocess.run(cmd, cwd=WAN_REPO_DIR, capture_output=True, text=True)

    if result.returncode != 0:
        print("\n[WAN GENERATION FAILED]")
        print(result.stdout)
        print(result.stderr)
    elif not os.path.exists(output_path):
        print("\n[ERROR] Command finished, but video file was not created.")
    else:
        print(f"Success! Video saved locally to {output_path}")
        if TESTING_MODE:
            display(Video(output_path, embed=True, width=720))

except Exception:
    print("\n[GENERAL ERROR IN WAN PIPELINE]")
    traceback.print_exc()


In [ ]:
!pip install audiocraft torchaudio

In [ ]:
# TEST C: AUDIOGEN
import os
import traceback
from IPython.display import Audio, display

from storage_manager import managed_storage_operation
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes

MODEL_ID = "facebook/audiogen-medium"
OUTPUT_DIR = os.path.join(core_config.SYSTEM_DIRS["outputs"], "tests")
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    import torchaudio
    from audiocraft.models import AudioGen
    from audiocraft.data.audio import audio_write

    print("Verifying AudioGen model in local cache...")
    try:
        model = AudioGen.get_pretrained(MODEL_ID)
        print("AudioGen cache hit or model already provisioned.")
    except Exception:
        required_gb = (
            estimate_hf_repo_bytes(MODEL_ID) / BYTES_PER_GB
            + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))
        )
        print(f"AudioGen cache incomplete. Storage admission budget: {required_gb:.2f} GB")
        with managed_storage_operation(
            "models",
            required_gb=required_gb,
            require_fresh_quota=True,
            minimum_free_gb=2.0,
        ):
            model = AudioGen.get_pretrained(MODEL_ID)

    model.set_generation_params(duration=5)
    descriptions = ["sci-fi spaceship airlock opening heavy hiss"]
    print(f"Generating audio for: {descriptions}")
    wav = model.generate(descriptions)

    output_name = "audiogen_test_0"
    audio_path = os.path.join(OUTPUT_DIR, output_name + ".wav")
    with managed_storage_operation(
        "outputs",
        required_gb=0.10,
        require_fresh_quota=False,
        minimum_free_gb=0.02,
    ):
        audio_write(
            os.path.splitext(audio_path)[0],
            wav[0].cpu(),
            model.sample_rate,
            strategy="loudness",
            loudness_compressor=True,
        )

    if os.path.exists(audio_path):
        print(f"Success! Audio saved to {audio_path}")
        if TESTING_MODE:
            display(Audio(audio_path, autoplay=False))
    else:
        print("[ERROR] Audio generation completed, but file was not saved.")

except ImportError:
    print("[ERROR] AudioCraft not installed properly. Provision dependencies on the compute node first.")
except Exception:
    print("\n[ERROR IN AUDIO PIPELINE]")
    traceback.print_exc()


In [ ]:
!pip install huggingface_hub

In [ ]:
# TEST D: CLOUD OFFLOADING (BATCH TRIGGER SIMULATION)
import os
import time
from huggingface_hub import HfApi

# Retrieve the token from secure memory
hf_token = os.environ.get("SECURE_HF_TOKEN")

if ENABLE_CLOUD_UPLOAD and hf_token:
    print(f"Initiating Cloud Upload Sequence (Trigger: {UPLOAD_TRIGGER_TYPE})...")
    
    # Replace 'your-username' with your actual Hugging Face username!
    repo_id = "strikerhell/a100-asset-vault-test" 
    
    # Create a dummy folder with "assets" to simulate a batch
    test_upload_dir = "batch_output_001"
    os.makedirs(test_upload_dir, exist_ok=True)
    
    # Simulate generating K=100 assets
    print(f"Simulating pipeline generating {BATCH_LIMIT} assets...")
    for i in range(5): # Just creating 5 files for a quick test
        with open(f"{test_upload_dir}/asset_{i}.txt", "w") as f:
            f.write("Generated asset data.")
    
    try:
        api = HfApi(token=hf_token)
        print(f"Ensuring private repo '{repo_id}' exists...")
        try:
            api.create_repo(repo_id=repo_id, repo_type="dataset", private=True)
        except Exception:
            pass # Repo already exists
            
        print(f"Uploading batch folder ({test_upload_dir}/) to HuggingFace Private Cloud...")
        api.upload_folder(
            folder_path=test_upload_dir,
            repo_id=repo_id,
            repo_type="dataset",
            path_in_repo=f"uploads/{test_upload_dir}" 
        )
        print("Upload Complete! The automated cloud batching is working.")
            
    except Exception as e:
        print("\n[ERROR IN CLOUD UPLOAD]")
        traceback.print_exc()
else:
    print("Cloud upload is currently DISABLED or Secure HF Token is missing in RAM.")

In [ ]:
!quota_check